# NYC Yellow Taxi Data Cleaning

## Import Libraries and Loading Data

In [45]:
import pandas as pd 
import numpy as np

In [46]:
df = pd.read_parquet('/home/sylviecalvaire/nyc-taxi/data/raw/yellow_tripdata_2026-02.parquet') 
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,7,2026-02-01 00:05:57,2026-02-01 00:05:57,1.00,0.94,1.00,N,107,170,1,7.20,0.00,0.50,0.00,0.00,1.00,12.95,2.50,0.00,0.75
1,7,2026-02-01 00:35:58,2026-02-01 00:35:58,1.00,1.93,1.00,N,234,141,1,11.40,0.00,0.50,3.43,0.00,1.00,20.58,2.50,0.00,0.75
2,2,2026-02-01 00:08:41,2026-02-01 00:39:32,1.00,9.99,1.00,N,138,68,1,44.30,6.00,0.50,11.01,0.00,1.00,67.81,2.50,1.75,0.75
3,1,2026-02-01 00:29:06,2026-02-01 00:41:04,0.00,1.70,1.00,N,209,13,1,12.80,4.25,0.50,3.70,0.00,1.00,22.25,2.50,0.00,0.75
4,1,2026-02-01 00:53:52,2026-02-01 01:11:21,0.00,3.70,1.00,N,249,229,1,19.80,4.25,0.50,6.35,0.00,1.00,31.90,2.50,0.00,0.75


## Initial Inspection

In [47]:
pd.set_option("display.float_format", "{:.2f}".format)
df.shape
df.columns
df.info()
df.describe(include="all")

## Duplicate Check

In [50]:
df.duplicated().sum()

np.int64(0)

## Missing Values

In [49]:
df.isnull().sum().sort_values(ascending=False)

passenger_count          1023317
congestion_surcharge     1023317
store_and_fwd_flag       1023317
RatecodeID               1023317
Airport_fee              1023317
tpep_dropoff_datetime          0
VendorID                       0
tpep_pickup_datetime           0
DOLocationID                   0
payment_type                   0
trip_distance                  0
PULocationID                   0
extra                          0
fare_amount                    0
mta_tax                        0
tip_amount                     0
improvement_surcharge          0
tolls_amount                   0
total_amount                   0
cbd_congestion_fee             0
dtype: int64

## Filling Missing Data

In [59]:
df["passenger_count"] = df["passenger_count"].fillna(0)
df["store_and_fwd_flag"] = df["store_and_fwd_flag"].fillna("Unknown")
df["RatecodeID"] = df["RatecodeID"].fillna(0)
df["congestion_surcharge"] = df["congestion_surcharge"].fillna(0)
df["Airport_fee"] = df["Airport_fee"].fillna(0)

## Invalid Record Filtering

In [51]:
df = df[df["trip_distance"] > 0]
df = df[df["fare_amount"] > 0]
df = df[df["total_amount"] > 0]

## Create New Features

In [52]:
df["trip_duration_min"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

In [56]:
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day_name"] = df["tpep_pickup_datetime"].dt.day_name()
df["pickup_day_of_week"] = df["tpep_pickup_datetime"].dt.dayofweek
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month
df["fare_per_mile"] = df["fare_amount"] / df["trip_distance"]

## Handle extreme outliers

In [54]:
df[["trip_distance", "fare_amount", "total_amount", "trip_duration_min"]].describe()

,trip_distance,fare_amount,total_amount,trip_duration_min
count,2268604.00,2268604.00,2268604.00,2268604.00
mean,3.40,19.67,29.20,17.63
std,23.44,17.97,22.34,15.47
min,0.01,0.01,1.01,0.02
25%,0.98,9.30,16.75,7.92
50%,1.65,13.50,21.90,13.13
75%,3.28,22.60,31.50,21.73
max,32775.53,1562.60,1580.14,299.90


In [55]:
df = df[df["trip_distance"] <= 100]
df = df[df["fare_amount"] <= 500]
df = df[df["total_amount"] <= 500]
df = df[df["trip_duration_min"] <= 300]

In [ ]:
df[["trip_distance", "fare_amount", "total_amount", "trip_duration_min"]].describe()

In [57]:
df.isnull().sum().sort_values(ascending=False)

VendorID                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
RatecodeID               0
store_and_fwd_flag       0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
congestion_surcharge     0
Airport_fee              0
cbd_congestion_fee       0
trip_duration_min        0
pickup_hour              0
pickup_day_name          0
pickup_day_of_week       0
pickup_month             0
fare_per_mile            0
dtype: int64

In [63]:
df.to_parquet("/home/sylviecalvaire/nyc-taxi/data/processed/yellow_tripdata_2026-02_cleaned.parquet", index=False)